# Thesis Figures Notebook

This notebook documents and generates the figures used in the current thesis draft `paper_artifacts/good_Paper_v1.md`.

Primary export directory: `paper_artifacts/figures_generated/`.

## Global notebook contract
- Run executable sections top-to-bottom from a clean kernel.
- Manual/prompt figures are documented here, but produced outside the notebook in Miro.com.
- Empirical figures are generated from repository artifacts under `final_runs/**` and must not rely on typed benchmark numbers except validation constants.
- `pnl_sum` / `net_pnl` is the primary deployment-oriented economic metric; `gross_pnl_sum`, `n_trades`, `trade_rate`, and `pnl_per_trade` explain signal extraction, turnover, selectivity, and cost drag.
- The notebook keeps one markdown description immediately before each figure-generation block, stating the figure's role in the thesis, what it shows, and what conclusion the text draws from it.


## Shared setup

The following cells define imports, thesis plotting style, export helpers, artifact paths, and shared artifact loaders used by all executable empirical figures.


In [ ]:
import os
import json
from pathlib import Path


def _find_repo_root_for_mpl(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "final_runs").exists() and (candidate / "paper_artifacts").exists():
            return candidate
    return start

_MPLCONFIGDIR = _find_repo_root_for_mpl() / "paper_artifacts" / ".mplconfig"
os.environ.setdefault("MPLCONFIGDIR", str(_MPLCONFIGDIR))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch, Circle


In [ ]:
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
    "grid.linestyle": "--",
})

THESIS_COLORS = {
    "navy": "#1f4e79",
    "teal": "#2a9d8f",
    "orange": "#f4a261",
    "slate": "#64748b",
    "ink": "#0f172a",
    "light_blue": "#e8f1fb",
    "light_teal": "#e5f4f1",
    "light_orange": "#fff0df",
    "light_slate": "#f1f5f9",
    "green": "#54a24b",
    "red": "#e45756",
    "purple": "#7c3aed",
}

FAMILY_COLORS = {
    "base-gnn": "#4c78a8",
    "multi-gnn": "#72b7b2",
    "memory-gnn": "#f58518",
}

RELATION_COLORS = {
    "price_dep": THESIS_COLORS["navy"],
    "order_flow": THESIS_COLORS["teal"],
    "liquidity": THESIS_COLORS["orange"],
}


def thesis_box(
    ax,
    xy,
    width,
    height,
    title,
    body="",
    facecolor=None,
    edgecolor="none",
    title_color=None,
    body_color="#334155",
    fontsize=9,
    title_size=10,
    align="center",
):
    """Draw a rounded publication-style annotation box."""
    facecolor = facecolor or THESIS_COLORS["light_slate"]
    title_color = title_color or THESIS_COLORS["ink"]
    patch = FancyBboxPatch(
        xy,
        width,
        height,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidth=0.8,
        alpha=0.98,
    )
    ax.add_patch(patch)
    x, y = xy
    ha = align
    tx = x + width / 2 if align == "center" else x + 0.025
    ax.text(tx, y + height * 0.66, title, ha=ha, va="center", fontsize=title_size, fontweight="bold", color=title_color)
    if body:
        ax.text(tx, y + height * 0.34, body, ha=ha, va="center", fontsize=fontsize, color=body_color, linespacing=1.25)
    return patch


def thesis_arrow(ax, start, end, color=None, lw=1.6, rad=0.0, mutation_scale=13, linestyle="-"):
    """Draw a consistent arrow for conceptual thesis diagrams."""
    arrow = FancyArrowPatch(
        start,
        end,
        arrowstyle="-|>",
        mutation_scale=mutation_scale,
        linewidth=lw,
        linestyle=linestyle,
        color=color or THESIS_COLORS["slate"],
        connectionstyle=f"arc3,rad={rad}",
        shrinkA=4,
        shrinkB=4,
    )
    ax.add_patch(arrow)
    return arrow


In [ ]:
def find_repo_root(start=None):
    """Find the repository root from any notebook working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "final_runs").exists() and (candidate / "paper_artifacts").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing final_runs/ and paper_artifacts/")


REPO_ROOT = find_repo_root()
THESIS_PATH = REPO_ROOT / "paper_artifacts" / "good_Paper_v1.md"
FIGURES_DIR = REPO_ROOT / "paper_artifacts" / "figures_generated"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_REGISTRY = []


def _rel(path):
    path = Path(path)
    try:
        return str(path.relative_to(REPO_ROOT))
    except ValueError:
        return str(path)


def save_thesis_figure(
    fig,
    figure_id,
    filename_stem,
    title,
    source_artifacts,
    rendering_method,
    main_text_section,
    status="implemented",
    validation_checks="Generated file is non-empty; semantic checks documented in notebook section.",
    save_vector=True,
):
    """Save a thesis figure and append a manifest row.

    PNG is used by the markdown thesis draft. SVG is retained for vector-quality
    export when Matplotlib can produce it.
    """
    png = FIGURES_DIR / f"{filename_stem}.png"
    svg = FIGURES_DIR / f"{filename_stem}.svg"
    fig.savefig(png, bbox_inches="tight", dpi=300)
    svg_value = ""
    if save_vector:
        fig.savefig(svg, bbox_inches="tight")
        svg_value = _rel(svg)
    # Keep repeated notebook execution from duplicating rows.
    FIGURE_REGISTRY[:] = [row for row in FIGURE_REGISTRY if row["figure_id"] != figure_id]
    FIGURE_REGISTRY.append({
        "figure_id": figure_id,
        "title": title,
        "filename_png": _rel(png),
        "filename_svg_or_pdf": svg_value,
        "rendering_method": rendering_method,
        "source_artifacts": "; ".join(_rel(pth) for pth in source_artifacts),
        "main_text_section": main_text_section,
        "status": status,
        "validation_checks": validation_checks,
    })
    return png


def write_manifest():
    manifest = pd.DataFrame(FIGURE_REGISTRY, columns=[
        "figure_id",
        "title",
        "filename_png",
        "filename_svg_or_pdf",
        "rendering_method",
        "source_artifacts",
        "main_text_section",
        "status",
        "validation_checks",
    ])
    csv_path = FIGURES_DIR / "figure_manifest.csv"
    md_path = FIGURES_DIR / "figure_manifest.md"
    manifest.to_csv(csv_path, index=False)
    rows = ["| " + " | ".join(manifest.columns) + " |", "| " + " | ".join(["---"] * len(manifest.columns)) + " |"]
    for _, row in manifest.fillna("").iterrows():
        rows.append("| " + " | ".join(str(row[col]).replace("|", "\\|") for col in manifest.columns) + " |")
    md_path.write_text("\n".join(rows) + "\n", encoding="utf-8")
    return manifest, csv_path, md_path


In [ ]:
SPLIT_PATHS = {
    "5min": REPO_ROOT / "final_runs/5min-base-gnn/splits/split_summary.json",
    "1min": REPO_ROOT / "final_runs/1min-base-gnn-conv/splits/split_summary.json",
    "1sec": REPO_ROOT / "final_runs/1sec-base-gnn-conv/splits/split_summary.json",
}
FINAL_HOLDOUT_ALIGNMENT_PATH = REPO_ROOT / "paper_artifacts/final_holdout_alignment_table.csv"

REGIME_SPECS = {
    "5min": {"run": "5min-base-gnn", "lookback": "30 min = 6 bars", "horizon": "5 min = 1 bar", "task_note": "same clock-time task as 1min"},
    "1min": {"run": "1min-base-gnn-conv", "lookback": "30 min = 30 bars", "horizon": "5 min = 5 bars", "task_note": "reference clock-time task"},
    "1sec": {"run": "1sec-base-gnn-conv", "lookback": "2 min = 120 bars", "horizon": "2 min = 120 bars", "task_note": "adapted high-frequency stress test"},
}


In [ ]:
PRIMARY_BENCHMARK_SOURCES = {
    ("5min", "base-gnn-conv"): (REPO_ROOT / "final_runs/5min-base-gnn/final_report.csv", "adaptive_conv"),
    ("5min", "base-gnn-mpnn"): (REPO_ROOT / "final_runs/5min-base-gnn/final_report.csv", "adaptive_mpnn"),
    ("5min", "multi-gnn-conv"): (REPO_ROOT / "final_runs/5min-multi-gnn/final_report.csv", "dynamic_rel_conv"),
    ("5min", "multi-gnn-mpnn"): (REPO_ROOT / "final_runs/5min-multi-gnn/final_report.csv", "dynamic_edge_mpnn"),
    ("5min", "memory-gnn-conv"): (REPO_ROOT / "final_runs/5min-memory-gnn/final_report.csv", "conv"),
    ("5min", "memory-gnn-mpnn"): (REPO_ROOT / "final_runs/5min-memory-gnn/final_report.csv", "mpnn"),
    ("1min", "base-gnn-conv"): (REPO_ROOT / "final_runs/1min-base-gnn-conv/final_report.csv", None),
    ("1min", "base-gnn-mpnn"): (REPO_ROOT / "final_runs/1min-base-gnn-mpnn/final_report.csv", None),
    ("1min", "multi-gnn-conv"): (REPO_ROOT / "final_runs/1min-multi-gnn-conv/final_report.csv", None),
    ("1min", "multi-gnn-mpnn"): (REPO_ROOT / "final_runs/1min-multi-gnn-mpnn/final_report.csv", None),
    ("1min", "memory-gnn-conv"): (REPO_ROOT / "final_runs/1min-memory-gnn/final_report.csv", "conv"),
    ("1min", "memory-gnn-mpnn"): (REPO_ROOT / "final_runs/1min-memory-gnn/final_report.csv", "mpnn"),
    ("1sec", "base-gnn-conv"): (REPO_ROOT / "final_runs/1sec-base-gnn-conv/final_report.csv", None),
    ("1sec", "base-gnn-mpnn"): (REPO_ROOT / "final_runs/1sec-base-gnn-mpnn/final_report.csv", None),
    ("1sec", "multi-gnn-conv"): (REPO_ROOT / "final_runs/1sec-multi-gnn-conv/final_report.csv", None),
    ("1sec", "multi-gnn-mpnn"): (REPO_ROOT / "final_runs/1sec-multi-gnn-mpnn/final_report.csv", None),
    ("1sec", "memory-gnn-conv"): (REPO_ROOT / "final_runs/1sec-memory-gnn-conv/final_report.csv", None),
    ("1sec", "memory-gnn-mpnn"): (REPO_ROOT / "final_runs/1sec-memory-gnn-mpnn/final_report.csv", None),
}

BENCHMARK_ORDER = [
    "base-gnn-conv",
    "base-gnn-mpnn",
    "multi-gnn-conv",
    "multi-gnn-mpnn",
    "memory-gnn-conv",
    "memory-gnn-mpnn",
]
FREQUENCY_ORDER = ["5min", "1min", "1sec"]
TRADE_COST = 0.0003


def load_split_summary(freq):
    with open(SPLIT_PATHS[freq], "r", encoding="utf-8") as f:
        return json.load(f)


def load_benchmark_metrics(model_state="last_cv", validate_cost=True):
    """Return the normalized benchmark table used by Figures 5.1, 5.4, D.1, and D.2."""
    rows = []
    for (freq, label), (csv_path, operator_filter) in PRIMARY_BENCHMARK_SOURCES.items():
        if not csv_path.exists():
            raise FileNotFoundError(csv_path)
        df = pd.read_csv(csv_path)
        df = df[df["model_state"] == model_state].copy()
        if operator_filter is not None and "operator" in df.columns:
            df = df[df["operator"] == operator_filter].copy()
        if df.empty:
            raise ValueError(f"No rows found for {freq} {label} ({model_state}) in {csv_path}")
        row = df.iloc[0].to_dict()
        row["frequency"] = freq
        row["model_label"] = label
        row["source_csv"] = csv_path
        rows.append(row)
    out = pd.DataFrame(rows)
    out["family"] = out["model_label"].str.extract(r"^(base-gnn|multi-gnn|memory-gnn)")
    out["operator_short"] = out["model_label"].str.extract(r"(conv|mpnn)$")[0].str.upper()
    out["x_label"] = out["family"].map({"base-gnn": "Base", "multi-gnn": "Multi", "memory-gnn": "Memory"}) + "\n" + out["operator_short"]
    out["order"] = out["model_label"].map({label: i for i, label in enumerate(BENCHMARK_ORDER)})
    out["frequency_order"] = out["frequency"].map({freq: i for i, freq in enumerate(FREQUENCY_ORDER)})
    out["cost_drag"] = out["gross_pnl_sum"] - out["pnl_sum"]
    out["implied_cost_drag"] = out["n_trades"] * TRADE_COST
    if validate_cost:
        cost_rows = out[["gross_pnl_sum", "pnl_sum", "n_trades"]].notna().all(axis=1)
        if cost_rows.any() and not np.allclose(out.loc[cost_rows, "cost_drag"], out.loc[cost_rows, "implied_cost_drag"], atol=1e-8):
            bad = out.loc[~np.isclose(out["cost_drag"], out["implied_cost_drag"], atol=1e-8), ["frequency", "model_label", "cost_drag", "implied_cost_drag"]]
            raise AssertionError(f"Benchmark cost-drag validation failed:\n{bad}")
    return out.sort_values(["frequency_order", "order"]).reset_index(drop=True)


# Backwards-compatible name used by earlier notebook cells.
def load_primary_benchmark_table(model_state="last_cv"):
    return load_benchmark_metrics(model_state=model_state)


## Figure 1.1 — Conceptual pipeline from LOB snapshots to graph-based entry decisions

**Purpose in the thesis**  
This opening figure introduces the study as a controlled end-to-end benchmark rather than an isolated prediction task. The text uses it to frame the full chain from multi-asset limit order book data to friction-aware final-holdout evaluation.

**What the figure shows**  
The figure should show a left-to-right pipeline: frequency-specific ADA/BTC/ETH LOB snapshots, feature construction, three-asset graph representation, graph model benchmark (`base_gnn`, `multigraph`, `memorygraph`, Conv/MPNN), entry-model outputs, and cost-aware final-holdout evaluation.

**Text interpretation**  
The main conclusion supported by the figure is that architectural comparison is meaningful only because data, targets, validation, thresholding, and backtest logic are held fixed; the thesis does not claim to depict a full production trading system.

**Metadata**
- Figure type: `conceptual`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 1
- Evidence source: `paper_artifacts/good_Paper_v1.md`
- Acceptance check: The diagram ends in post-cost final-holdout evaluation and visually preserves the controlled-benchmark framing.


## Figure 1.2 — Research-question map for the controlled graph benchmark

**Purpose in the thesis**  
This figure organizes RQ1–RQ4 before the methods table. The text uses it to show that the questions are coordinated parts of one benchmark design rather than four disconnected experiments.

**What the figure shows**  
The figure maps the four research questions to the four experimental dimensions: model family (`base_gnn`, `multigraph`, `memorygraph`), graph operator (Conv versus MPNN), temporal resolution (5min, 1min, 1sec), and deployment-oriented model state (`last_CV` versus `final_refit`). A shared benchmark core should connect all four branches.

**Text interpretation**  
The figure supports the thesis logic that model-family, operator, frequency, and model-state comparisons are interpreted under common data, target construction, validation, and event-based backtest rules.

**Metadata**
- Figure type: `conceptual`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 1
- Evidence source: `paper_artifacts/good_Paper_v1.md`
- Acceptance check: Each RQ is linked to its benchmark dimension and the shared controlled-evaluation core remains explicit.


## Figure 3.1 — Graph input representation for the three-asset LOB benchmark

**Purpose in the thesis**  
This methodology figure formalizes the common graph input shared by all model families. The text uses it to justify that later architectural comparisons differ in graph processing, not in the input universe.

**What the figure shows**  
The figure should show ADA, BTC, and ETH as fixed nodes in a directed complete graph with self-loops. ETH is marked as the supervised target asset. Edge semantics are represented through the three relation channels `price_dep`, `order_flow`, and `liquidity`, while side annotations explain dynamic node states and relation-aware edge states over the lookback window.

**Text interpretation**  
The figure supports the statement that every architecture receives the same three-node graph-structured input: dynamic node features, relation-aware edge channels, and an ETH-centered target.

**Metadata**
- Figure type: `conceptual-methodological`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 3
- Evidence source: `paper_artifacts/good_Paper_v1.md`
- Acceptance check: Directed completeness, self-loops, ETH target status, and the three relation channels are all visible.


## Figure 3.2 — Frequency regimes and final-holdout split design

**Purpose in the thesis**  
This figure explains the sample-design difference between the shared 5min/1min task and the adapted 1sec stress test. The text uses it to prevent over-interpreting the 1sec regime as a perfectly symmetric continuation of the lower-frequency task.

**What the figure shows**  
The plot summarizes each frequency's working interval, pre-holdout region, purge gap, and final holdout. It also compares lookback and horizon settings: 5min and 1min share the same 30-minute lookback and five-minute horizon, while 1sec uses a two-minute lookback and two-minute horizon.

**Text interpretation**  
The figure supports the conclusion that 5min and 1min are directly comparable as a shared clock-time benchmark, whereas 1sec is a frequency-adapted high-frequency stress test.

**Metadata**
- Figure type: `empirical-design`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 3
- Primary inputs: `final_runs/*/splits/split_summary.json`, `paper_artifacts/final_holdout_alignment_table.csv`
- Acceptance check: All three frequencies, holdout segments, purge gaps, and frequency-specific horizon/lookback settings are visible.


In [ ]:
split_summaries = {freq: load_split_summary(freq) for freq in SPLIT_PATHS}
alignment_df = pd.read_csv(FINAL_HOLDOUT_ALIGNMENT_PATH)

regime_rows = []
for freq in ["5min", "1min", "1sec"]:
    summary = split_summaries[freq]
    spec = REGIME_SPECS[freq]
    artifact_row = alignment_df.loc[alignment_df["run"] == spec["run"]].iloc[0]
    work_start = float(artifact_row["data_slice_start_frac"])
    work_end = float(artifact_row["data_slice_end_frac"])
    work_width = work_end - work_start
    preholdout_end_full_frac = work_start + work_width * (float(artifact_row["preholdout_n"]) / float(artifact_row["n_samples"]))
    holdout_start_full_frac = float(artifact_row["effective_holdout_start_full_frac"])
    holdout_end_full_frac = float(artifact_row["effective_holdout_end_full_frac"])

    regime_rows.append({
        "frequency": freq,
        "run": spec["run"],
        "work_start": work_start,
        "work_end": work_end,
        "preholdout_end_full_frac": preholdout_end_full_frac,
        "holdout_start_full_frac": holdout_start_full_frac,
        "holdout_end_full_frac": holdout_end_full_frac,
        "holdout_frac_of_slice": float(artifact_row["final_holdout_frac"]),
        "lookback": spec["lookback"],
        "horizon": spec["horizon"],
        "task_note": spec["task_note"],
        "cv_folds": int(summary["num_train_folds"]),
        "purge_gap_bars": int(summary["purge_gap_bars"]),
        "holdout_start_utc": artifact_row["holdout_start_utc"],
        "holdout_end_utc": artifact_row["holdout_end_utc"],
        "start_delta_vs_1min_sec": int(artifact_row["start_delta_vs_1min_sec"]),
        "end_delta_vs_1min_sec": int(artifact_row["end_delta_vs_1min_sec"]),
    })

regime_df = pd.DataFrame(regime_rows)
regime_df


In [ ]:
regime_colors = {
    "outside": "#e5e7eb",
    "working": "#dbeafe",
    "preholdout": "#4c78a8",
    "purge_gap": "#cbd5e1",
    "holdout": "#f58518",
}


In [ ]:
fig, axes = plt.subplots(
    2,
    1,
    figsize=(12.4, 7.4),
    constrained_layout=True,
    height_ratios=[1.3, 1.05],
)

ax = axes[0]
y_positions = np.arange(len(regime_df))[::-1]
bar_height = 0.34

for y, row in zip(y_positions, regime_df.to_dict("records")):
    ax.add_patch(Rectangle((0.0, y - bar_height / 2), 1.0, bar_height, facecolor=regime_colors["outside"], edgecolor="none", zorder=0))
    ax.add_patch(Rectangle((row["work_start"], y - bar_height / 2), row["work_end"] - row["work_start"], bar_height, facecolor=regime_colors["working"], edgecolor="white", linewidth=0.9, zorder=1))
    ax.add_patch(Rectangle((row["work_start"], y - bar_height / 2), row["preholdout_end_full_frac"] - row["work_start"], bar_height, facecolor=regime_colors["preholdout"], edgecolor="white", linewidth=0.8, zorder=2))
    ax.add_patch(Rectangle((row["preholdout_end_full_frac"], y - bar_height / 2), row["holdout_start_full_frac"] - row["preholdout_end_full_frac"], bar_height, facecolor=regime_colors["purge_gap"], edgecolor="white", linewidth=0.6, hatch="///", zorder=3))
    ax.add_patch(Rectangle((row["holdout_start_full_frac"], y - bar_height / 2), row["holdout_end_full_frac"] - row["holdout_start_full_frac"], bar_height, facecolor=regime_colors["holdout"], edgecolor="white", linewidth=0.8, zorder=4))
    ax.text(-0.018, y, row["frequency"], va="center", ha="right", fontweight="bold", fontsize=10, color="#0f172a")
    ax.text(row["work_start"] + 0.004, y + 0.26, f"slice {row['work_start']:.0%}-{row['work_end']:.0%}", fontsize=8, ha="left", va="center", color="#334155")
    ax.text(row["holdout_start_full_frac"] + 0.004, y, f"holdout\n{row['holdout_frac_of_slice']:.1%} of slice", va="center", ha="left", fontsize=7.8, color="#7c2d12")
    ax.text(row["work_start"] + 0.004, y - 0.28, f"{row['lookback']} | {row['horizon']} | folds: {row['cv_folds']} | purge gap: {row['purge_gap_bars']} bars | {row['task_note']}", fontsize=8, ha="left", va="center", color="#334155")

legend_handles = [
    Rectangle((0, 0), 1, 1, facecolor=regime_colors["outside"], edgecolor="none", label="outside working slice"),
    Rectangle((0, 0), 1, 1, facecolor=regime_colors["preholdout"], edgecolor="none", label="pre-holdout development region"),
    Rectangle((0, 0), 1, 1, facecolor=regime_colors["purge_gap"], edgecolor="none", hatch="///", label="purge gap before final holdout"),
    Rectangle((0, 0), 1, 1, facecolor=regime_colors["holdout"], edgecolor="none", label="final blind holdout"),
]
ax.legend(handles=legend_handles, loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=4, frameon=False, fontsize=8)
ax.set_xlim(0, 1.0)
ax.set_ylim(-0.6, len(regime_df) - 0.1)
ax.set_yticks([])
ax.set_xticks(np.linspace(0, 1, 6))
ax.set_xticklabels([f"{int(x * 100)}%" for x in np.linspace(0, 1, 6)])
ax.set_xlabel("Full series position")
ax.set_title("Figure 3.2 — Frequency regimes and final-holdout split design")
ax.text(0.0, len(regime_df) - 0.02, "Top panel: actual full-series positions from saved split artifacts", ha="left", va="bottom", fontsize=10, fontweight="bold")

ax2 = axes[1]
mdates = plt.matplotlib.dates
holdout_rows = regime_df.copy()
holdout_rows["holdout_start_ts"] = pd.to_datetime(holdout_rows["holdout_start_utc"], utc=True)
holdout_rows["holdout_end_ts"] = pd.to_datetime(holdout_rows["holdout_end_utc"], utc=True)
reference_row = holdout_rows.loc[holdout_rows["frequency"] == "1min"].iloc[0]
ref_start = reference_row["holdout_start_ts"]
ref_end = reference_row["holdout_end_ts"]

for y, row in zip(y_positions, holdout_rows.to_dict("records")):
    start_ts = pd.Timestamp(row["holdout_start_ts"])
    end_ts = pd.Timestamp(row["holdout_end_ts"])
    ax2.plot([start_ts, end_ts], [y, y], color=regime_colors["holdout"], linewidth=10, solid_capstyle="butt")
    ax2.scatter([start_ts, end_ts], [y, y], s=22, color="#9a3412", zorder=3)
    delta_label = f"start {row['start_delta_vs_1min_sec']:+d}s | end {row['end_delta_vs_1min_sec']:+d}s"
    ax2.text(end_ts + pd.Timedelta(minutes=18), y, delta_label, va="center", ha="left", fontsize=8, color="#7c2d12")

ax2.axvline(ref_start, color="#475569", linestyle="--", linewidth=1.0)
ax2.axvline(ref_end, color="#475569", linestyle="--", linewidth=1.0)
ax2.text(ref_start, len(holdout_rows) - 0.15, "1min start", ha="left", va="bottom", fontsize=8, color="#475569")
ax2.text(ref_end, len(holdout_rows) - 0.15, "1min end", ha="right", va="bottom", fontsize=8, color="#475569")

xmin = holdout_rows["holdout_start_ts"].min() - pd.Timedelta(minutes=20)
xmax = holdout_rows["holdout_end_ts"].max() + pd.Timedelta(minutes=65)
ax2.set_xlim(xmin, xmax)
ax2.set_ylim(-0.55, len(holdout_rows) - 0.1)
ax2.set_yticks(y_positions)
ax2.set_yticklabels(holdout_rows["frequency"])
ax2.xaxis.set_major_locator(mdates.HourLocator(interval=6, tz=holdout_rows["holdout_start_ts"].iloc[0].tz))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M", tz=holdout_rows["holdout_start_ts"].iloc[0].tz))
ax2.set_title("Bottom panel: actual calendar-time final holdout windows", loc="left", fontsize=10, fontweight="bold")
ax2.set_xlabel("UTC timestamp")
ax2.text(mdates.date2num(xmin), -0.42, "1min and 5min are effectively aligned; 1sec starts 34s later and ends 181s later while targeting the same late-period market segment.", ha="left", va="bottom", fontsize=8.8, color="#334155")

fig


In [ ]:
fig_3_2_path = save_thesis_figure(
    fig,
    figure_id="3.2",
    filename_stem="fig_3_2_split_design_v2",
    title="Frequency regimes and final-holdout split design",
    source_artifacts=['paper_artifacts/final_holdout_alignment_table.csv', 'final_runs/*/splits/split_summary.json'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 3",
    validation_checks="Frequency regimes and aligned final-holdout intervals are artifact-grounded.",
)
fig_3_2_path


## Figure 3.3 — Triple-barrier target construction for the ETH midpoint

**Purpose in the thesis**  
This figure explains how the shared learning targets are built from future ETH midpoint paths. The text uses it to make the target-generation logic transparent before model training and backtesting are discussed.

**What the figure shows**  
The plot shows an ETH midpoint path from an anchor time, volatility-scaled upper and lower barriers, a vertical horizon barrier, and the first barrier/event that determines realized return, direction, trade relevance, exit type, and time-to-event.

**Text interpretation**  
The figure supports the claim that all model families are trained on the same path-dependent triple-barrier target construction, which is why purging is required in the walk-forward validation design.

**Metadata**
- Figure type: `methodological-example`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 3
- Primary inputs: ETH midpoint data and target-construction parameters used by the benchmark pipeline
- Acceptance check: Upper/lower/vertical barriers and the realized first-hit event are visually identifiable.


In [ ]:
triple_barrier_config = {
    "freq": "1min",
    "lookback_bars": 30,
    "horizon_bars": int(5),
    "upper_barrier_bps": 8.0,
    "lower_barrier_bps": 8.0,
    "vol_lookback_bars": 30,
    "vol_barrier_mult_up": 1.8,
    "vol_barrier_mult_down": 1.8,
    "min_barrier_bps": 4.0,
    "max_barrier_bps": 30.0,
    "trade_label_buffer_bps": 0.5,
    "cost_bps_per_side": 1.0,
    "execution_cost_multiplier": 1.0,
    "use_cost_in_label": True,
}

eth_1min_df = pd.read_csv(REPO_ROOT / "../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1min.csv")
eth_1min_df["system_time"] = pd.to_datetime(eth_1min_df["system_time"])
eth_mid = eth_1min_df["midpoint"].to_numpy(dtype=float)
eth_log_mid = np.log(eth_mid + 1e-12)
eth_lr_1bar = np.zeros(len(eth_mid), dtype=float)
eth_lr_1bar[1:] = np.diff(eth_log_mid)


In [ ]:
trade_edge_threshold = (
    triple_barrier_config["trade_label_buffer_bps"] * 1e-4
    + (3.0 * triple_barrier_config["cost_bps_per_side"] * 1e-4) * triple_barrier_config["execution_cost_multiplier"]
)

rolling_vol = pd.Series(eth_lr_1bar).rolling(
    window=triple_barrier_config["vol_lookback_bars"],
    min_periods=max(3, triple_barrier_config["vol_lookback_bars"] // 3),
).std().to_numpy(dtype=float)
vol_bps = np.abs(rolling_vol) * 1e4
upper_bps = np.clip(
    vol_bps * triple_barrier_config["vol_barrier_mult_up"],
    triple_barrier_config["min_barrier_bps"],
    triple_barrier_config["max_barrier_bps"],
)
lower_bps = np.clip(
    vol_bps * triple_barrier_config["vol_barrier_mult_down"],
    triple_barrier_config["min_barrier_bps"],
    triple_barrier_config["max_barrier_bps"],
)
upper_bps = np.where(np.isfinite(upper_bps), upper_bps, triple_barrier_config["upper_barrier_bps"])
lower_bps = np.where(np.isfinite(lower_bps), lower_bps, triple_barrier_config["lower_barrier_bps"])


In [ ]:
triple_barrier_candidates = []
for t in range(triple_barrier_config["lookback_bars"], len(eth_mid) - triple_barrier_config["horizon_bars"] - 1):
    upper_lr = max(float(upper_bps[t]) * 1e-4, 1e-8)
    lower_lr = max(float(lower_bps[t]) * 1e-4, 1e-8)
    future_path = eth_log_mid[t + 1 : t + triple_barrier_config["horizon_bars"] + 1] - eth_log_mid[t]
    if len(future_path) == 0 or not np.isfinite(future_path).all():
        continue

    upper_hits = np.where(future_path >= upper_lr - 1e-12)[0]
    lower_hits = np.where(future_path <= -lower_lr + 1e-12)[0]
    first_upper = int(upper_hits[0]) + 1 if upper_hits.size else None
    first_lower = int(lower_hits[0]) + 1 if lower_hits.size else None

    if first_upper is not None and (first_lower is None or first_upper <= first_lower):
        exit_type = "upper"
        tte = first_upper
        realized_return = upper_lr
        direction_label = "long"
        trade_label = 1.0 if upper_lr > trade_edge_threshold else 0.0
    elif first_lower is not None and (first_upper is None or first_lower < first_upper):
        exit_type = "lower"
        tte = first_lower
        realized_return = -lower_lr
        direction_label = "short"
        trade_label = 1.0 if lower_lr > trade_edge_threshold else 0.0
    else:
        exit_type = "vertical"
        tte = triple_barrier_config["horizon_bars"]
        realized_return = float(future_path[-1])
        direction_label = "long" if realized_return > 0 else "short"
        trade_label = 0.0

    if trade_label > 0.5 and exit_type in {"upper", "lower"}:
        score = abs(tte - max(2, triple_barrier_config["horizon_bars"] // 2)) + 0.1 * abs(abs(realized_return) - trade_edge_threshold)
        triple_barrier_candidates.append((score, t, exit_type, tte, realized_return, direction_label, upper_lr, lower_lr))

_, target_idx, exit_type, tte, realized_return, direction_label, upper_lr, lower_lr = sorted(triple_barrier_candidates, key=lambda x: x[0])[0]
base_midpoint = eth_mid[target_idx]
path_prices = eth_mid[target_idx : target_idx + triple_barrier_config["horizon_bars"] + 1]
time_axis = np.arange(0, triple_barrier_config["horizon_bars"] + 1)
upper_price = base_midpoint * np.exp(upper_lr)
lower_price = base_midpoint * np.exp(-lower_lr)
exit_price = upper_price if exit_type == "upper" else lower_price if exit_type == "lower" else path_prices[-1]
eth_1min_df.loc[[target_idx], ["system_time", "midpoint"]]


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8), constrained_layout=True)
ax.plot(time_axis, path_prices, color="#1f2937", linewidth=2.2, marker="o", markersize=4, label="ETH midpoint path")
ax.axhline(base_midpoint, color="#94a3b8", linewidth=1.2, linestyle="--", label="entry midpoint")
ax.axhline(upper_price, color="#16a34a", linewidth=1.6, linestyle="--", label="upper barrier")
ax.axhline(lower_price, color="#dc2626", linewidth=1.6, linestyle="--", label="lower barrier")
ax.axvline(triple_barrier_config["horizon_bars"], color="#7c3aed", linewidth=1.6, linestyle=":", label="vertical barrier")
ax.scatter([tte], [exit_price], color="#f59e0b", s=80, zorder=5, edgecolor="black", linewidth=0.5, label="realized exit")
ax.annotate("realized exit", xy=(tte, exit_price), xytext=(tte + 0.35, exit_price + (upper_price - lower_price) * 0.15), arrowprops=dict(arrowstyle="->", lw=1.1, color="#475569"), fontsize=9)
ax.set_title("Figure 3.3 — Triple-barrier target construction for the ETH midpoint")
ax.set_xlabel("Bars after target timestamp (1min frequency)")
ax.set_ylabel("ETH midpoint")
ax.legend(loc="upper left", frameon=False, ncol=2)

info_text = (
    f"timestamp: {eth_1min_df.loc[target_idx, 'system_time']}\n"
    f"exit type: {exit_type}\n"
    f"direction label: {direction_label}\n"
    f"trade relevance label: 1\n"
    f"tte: {tte} bars\n"
    f"realized return: {realized_return * 1e4:.2f} bps"
)
ax.text(1.02, 0.95, info_text, transform=ax.transAxes, va="top", fontsize=9, bbox=dict(boxstyle="round,pad=0.4", facecolor="#f8fafc", edgecolor="#cbd5e1"))
ax.text(0.01, -0.18, "Selection rule: first 1min ETH example after the 30-bar volatility warm-up with a trade-relevant non-vertical event and a time-to-exit closest to the middle of the 5-bar horizon.", transform=ax.transAxes, fontsize=8, color="#475569")
fig


In [ ]:
fig_3_3_path = save_thesis_figure(
    fig,
    figure_id="3.3",
    filename_stem="fig_3_3_triple_barrier_eth_midpoint",
    title="Triple-barrier target construction for the ETH midpoint",
    source_artifacts=['../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1min.csv', 'final_runs/1min-base-gnn-conv/resolved_config.yaml'],
    rendering_method="Python-primary hybrid",
    main_text_section="Chapter 3",
    validation_checks="Upper/lower/vertical barriers and realized event exit are visible.",
)
fig_3_3_path


## Figure 3.4 — Common entry-model backtest and post-cost PnL calculation

**Purpose in the thesis**  
This figure explains the shared trading-evaluation rule that converts model outputs into comparable final-holdout evidence. The text uses it to show why differences in results are not caused by family-specific backtest logic.

**What the figure shows**  
The figure should show the evaluation flow from model heads to trade activation, direction selection, sequential non-overlapping event-based positions, realized exit, gross PnL, transaction-cost proxy, and net PnL. A compact formula panel should include `gross PnL_i = s_i r_i`, `net PnL_i = gross PnL_i - c_rt`, and the benchmark round-trip cost proxy.

**Text interpretation**  
The figure supports the conclusion that the benchmark is cost-aware and fair across model families, while remaining a transparent entry-model evaluation rather than a full execution simulator.

**Metadata**
- Figure type: `conceptual-methodological`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 3
- Evidence source: `paper_artifacts/good_Paper_v1.md`, `train.py`, final-holdout trade logs
- Acceptance check: Trade activation, direction, event exit, gross-to-net conversion, and the shared cost rule are explicit.


## Figure 3.5 — Purged walk-forward validation and deployment-oriented model states

**Purpose in the thesis**  
This figure explains the chronological validation design and the difference between the two model states used later in Chapter 5. The text uses it to justify `last_CV` as the primary deployment-oriented reference and `final_refit` as an informative robustness state.

**What the figure shows**  
The plot shows chronological train, purge, validation, purge, test, pre-holdout, and final-holdout segments. It also indicates how the final walk-forward fold produces the `last_CV` model and how a larger pre-holdout refit produces the `final_refit` model.

**Text interpretation**  
The figure supports the claim that path-dependent labels require purged chronological validation and that `last_CV` and `final_refit` are related but not interchangeable deployment states.

**Metadata**
- Figure type: `empirical-design`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 3
- Primary inputs: `final_runs/*/splits/split_summary.json`
- Acceptance check: Purge gaps are visible and both `last_CV` and `final_refit` model-state origins are labelled.


In [ ]:
walkforward_summary = load_split_summary("1min")
preholdout = walkforward_summary["preholdout"]
holdout = walkforward_summary["holdout"]
cv_folds = walkforward_summary["cv_folds"]
last_fold = cv_folds[-1]

full_total = preholdout["n_samples"] + holdout["n_samples"]
preholdout_fraction = preholdout["n_samples"] / full_total
holdout_fraction = holdout["n_samples"] / full_total


In [ ]:
train_n = last_fold["train"]["n_samples"]
validation_n = last_fold["val"]["n_samples"]
test_n = last_fold["test"]["n_samples"]
purge_gap = walkforward_summary["purge_gap_bars"]
fold_total = train_n + validation_n + test_n + 2 * purge_gap

fold_segments = [
    ("train", train_n / fold_total, "#4c78a8"),
    ("purge", purge_gap / fold_total, "#cbd5e1"),
    ("validation", validation_n / fold_total, "#72b7b2"),
    ("purge", purge_gap / fold_total, "#cbd5e1"),
    ("test", test_n / fold_total, "#54a24b"),
]


In [ ]:
fig = plt.figure(figsize=(11.5, 7.2), constrained_layout=True)
grid = fig.add_gridspec(3, 1, height_ratios=[1, 1, 1.15])
ax1 = fig.add_subplot(grid[0])
ax2 = fig.add_subplot(grid[1])
ax3 = fig.add_subplot(grid[2])

# top panel: global experiment timeline
ax1.add_patch(Rectangle((0, 0.35), preholdout_fraction, 0.3, facecolor="#4c78a8", edgecolor="white"))
ax1.add_patch(Rectangle((preholdout_fraction, 0.35), holdout_fraction, 0.3, facecolor="#f58518", edgecolor="white"))
ax1.text(preholdout_fraction / 2, 0.5, "pre-holdout\n(model development)", ha="center", va="center", color="white", fontweight="bold")
ax1.text(preholdout_fraction + holdout_fraction / 2, 0.5, "final holdout\n(blind evaluation)", ha="center", va="center", color="white", fontweight="bold")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.set_yticks([])
ax1.set_xticks([0, preholdout_fraction, 1.0])
ax1.set_xticklabels(["start", "holdout begins", "end"])
ax1.set_title("Figure 3.5 — Purged walk-forward validation and deployment-oriented model states")
ax1.text(0.0, 0.86, "Experiment timeline (1min regime shown as representative chronological benchmark)", fontsize=10, fontweight="bold", ha="left")

# middle panel: representative final CV fold
ax2.axis("off")
current_x = 0.02
for label, width, color in fold_segments:
    scaled_width = width * 0.96
    ax2.add_patch(Rectangle((current_x, 0.36), scaled_width, 0.28, facecolor=color, edgecolor="white"))
    ax2.text(current_x + scaled_width / 2, 0.5, label, ha="center", va="center", fontsize=9, fontweight="bold" if label in {"train", "validation", "test"} else None)
    current_x += scaled_width
ax2.text(0.02, 0.82, f"Representative final CV fold | purge gap = {purge_gap} bars | folds = {walkforward_summary['num_train_folds']}", fontsize=10, fontweight="bold")
ax2.text(0.02, 0.16, "Chronology is preserved and leakage is reduced by inserting purge gaps around validation and test boundaries.", fontsize=9)

# bottom panel: model states
ax3.axis("off")
state_boxes = {
    "best_CV": (0.08, 0.58, 0.22, 0.22, "#72b7b2", "best_CV\nstrongest selected CV checkpoint"),
    "last_CV": (0.39, 0.58, 0.22, 0.22, "#4c78a8", "last_CV\nfinal chronological fold model"),
    "final_refit": (0.70, 0.58, 0.22, 0.22, "#f58518", "final_refit\nrefit on largest pre-holdout sample"),
}
for x, y, w, h, color, label in state_boxes.values():
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.03", facecolor=color, edgecolor="none", alpha=0.96)
    ax3.add_patch(patch)
    ax3.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=9, color="white", fontweight="bold")

ax3.annotate("", xy=(0.19, 0.58), xytext=(0.22, 0.38), arrowprops=dict(arrowstyle="->", lw=1.4, color="#475569"))
ax3.annotate("", xy=(0.50, 0.58), xytext=(0.50, 0.38), arrowprops=dict(arrowstyle="->", lw=1.4, color="#475569"))
ax3.annotate("", xy=(0.81, 0.58), xytext=(0.78, 0.38), arrowprops=dict(arrowstyle="->", lw=1.4, color="#475569"))
ax3.text(0.10, 0.28, "selected from CV candidates", ha="center", fontsize=9)
ax3.text(0.50, 0.28, "deployment-primary reference", ha="center", fontsize=9)
ax3.text(0.80, 0.28, "diagnostic larger-sample refit", ha="center", fontsize=9)
ax3.text(0.02, 0.92, "Deployment-oriented model states", fontsize=10, fontweight="bold")

fig


In [ ]:
fig_3_5_path = save_thesis_figure(
    fig,
    figure_id="3.5",
    filename_stem="fig_3_5_walkforward_model_states",
    title="Purged walk-forward validation and deployment-oriented model states",
    source_artifacts=['final_runs/*/splits/split_summary.json'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 3",
    validation_checks="Chronological folds, purges, final holdout, and model states are shown.",
)
fig_3_5_path


## Figure 4.1 — Architecture comparison of `base_gnn`, `multigraph`, and `memorygraph`

**Purpose in the thesis**  
This chapter-opening figure summarizes the controlled architectural ablation. The text uses it to show that the three families share inputs and outputs but differ in relation handling, graph pathway structure, temporal mechanism, and statefulness.

**What the figure shows**  
The figure should present three parallel columns for `base_gnn`, `multigraph`, and `memorygraph`, each starting from shared graph inputs and ending in shared multi-task heads. The comparison strip should emphasize early relation fusion and a single graph pathway for `base_gnn`, relation-specific graph pathways and late fusion for `multigraph`, and recurrent node-edge memory for `memorygraph`.

**Text interpretation**  
The figure supports the conclusion that Chapter 4 compares architectures under a fixed benchmark, not models trained under different information sets.

**Metadata**
- Figure type: `conceptual-architecture`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 4
- Evidence source: `paper_artifacts/good_Paper_v1.md`, model-family pipeline files
- Acceptance check: Shared inputs/outputs are visually common; family-specific relation and temporal mechanisms are visually distinct.


## Figure 4.2 — Detailed architecture of the `base_gnn` family

**Purpose in the thesis**  
This figure explains why `base_gnn` is the clean single-graph baseline. The text uses it to define the early-fusion alternative against which the more complex relation-preserving and recurrent families are compared.

**What the figure shows**  
The figure should show parallel node and edge temporal encoders, `HybridEdgeFeatureFusion`, `EdgeRelationFusion`, one fused edge representation, a single graph operator block, graph readout, target temporal trunk or output projection, and the shared multi-task heads.

**Text interpretation**  
The figure supports the conclusion that `base_gnn` compresses relation channels before message passing and then tests whether one fused graph pathway is sufficient under the shared benchmark.

**Metadata**
- Figure type: `conceptual-architecture`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 4
- Evidence source: `paper_artifacts/good_Paper_v1.md`, `models/base_gnn_pipeline.py`
- Acceptance check: Early relation fusion is visibly before graph processing and only one graph pathway remains after fusion.


## Figure 4.3 — Detailed architecture of the `multigraph` family

**Purpose in the thesis**  
This figure makes the contrast with `base_gnn` visually explicit. The text uses it to explain the late-fusion hypothesis: relation semantics may be more useful if they remain separate during message passing.

**What the figure shows**  
The figure should show node and edge temporal encoders, `HybridEdgeFeatureFusion`, three relation-specific graph lanes for `price_dep`, `order_flow`, and `liquidity`, `RelationAttentionFusion`, graph readout, target temporal trunk or output projection, and the shared multi-task heads.

**Text interpretation**  
The figure supports the conclusion that `multigraph` preserves relation-specific semantics through graph updates and only then merges them through learned relation attention.

**Metadata**
- Figure type: `conceptual-architecture`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 4
- Evidence source: `paper_artifacts/good_Paper_v1.md`, `models/multigraph_pipeline.py`
- Acceptance check: The three relation lanes are equally visible and fusion happens after relation-specific graph processing.


## Figure 4.4 — Detailed architecture of the `memorygraph` family

**Purpose in the thesis**  
This figure explains why `memorygraph` is qualitatively different from the two convolutional temporal families. The text uses it to highlight recurrent statefulness rather than another static relation-fusion variant.

**What the figure shows**  
The figure should use a loop-based layout with current `X_node_t` and `X_edge_t`, step projectors, `HybridEdgeFeatureFusion`, previous edge and node memory states, edge-memory update, graph operator inside the recurrent loop, node-memory update, updated memories carried forward, graph readout, output projection, and shared heads.

**Text interpretation**  
The figure supports the conclusion that temporal modelling in `memorygraph` is represented through recurrent node-edge memory and that graph interaction occurs inside the recurrent update loop.

**Metadata**
- Figure type: `conceptual-architecture`
- Rendering method: `prompt+manual in Miro.com`
- Main text section: Chapter 4
- Evidence source: `paper_artifacts/good_Paper_v1.md`, `models/memorygraph_pipeline.py`
- Acceptance check: Previous/current/updated memory states are distinguishable, edge memory updates before node memory, and the graph operator is inside the loop.


## Figure 5.1 — Benchmark overview by frequency, graph family, and operator

**Purpose in the thesis**  
This figure gives the visual overview of the main `last_CV` benchmark before the exact values in Table 5.1. The text uses it to orient the reader across all eighteen model-frequency configurations.

**What the figure shows**  
The heatmap displays post-cost net PnL (`net_pnl` / `pnl_sum`) for each frequency, graph family, and graph operator, with trade counts annotated so that endpoint performance can be read together with activity level.

**Text interpretation**  
The figure supports the main ranking pattern: `base_gnn` with the Conv operator is strongest at 5min and 1min, while all 1sec configurations remain negative after transaction costs despite some positive gross signal.

**Metadata**
- Figure type: `empirical-result`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 5
- Primary inputs: `final_runs/*/final_report.csv`
- Acceptance check: Eighteen `last_CV` configurations are present and net PnL is the visual metric.


In [ ]:
benchmark_df = load_benchmark_metrics(model_state="last_cv")
benchmark_df[["frequency", "model_label", "pnl_sum", "gross_pnl_sum", "n_trades", "trade_rate", "pnl_per_trade", "cost_drag"]]


In [ ]:
benchmark_panels = {
    freq: benchmark_df[benchmark_df["frequency"] == freq].sort_values("order").reset_index(drop=True)
    for freq in FREQUENCY_ORDER
}


In [ ]:
heat = benchmark_df.pivot(index="model_label", columns="frequency", values="pnl_sum").loc[BENCHMARK_ORDER, FREQUENCY_ORDER]
trade_counts = benchmark_df.pivot(index="model_label", columns="frequency", values="n_trades").loc[BENCHMARK_ORDER, FREQUENCY_ORDER]
vmax = float(np.nanmax(np.abs(heat.to_numpy())))
norm = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

# Use an explicit mesh instead of imshow so saved vector/raster outputs preserve
# the intended rectangular cells rather than drifting toward square blocks.
fig = plt.figure(figsize=(9.4, 5.8), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[18, 1])
ax = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

x_edges = np.arange(len(FREQUENCY_ORDER) + 1)
y_edges = np.arange(len(BENCHMARK_ORDER) + 1)
mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    heat.to_numpy(),
    cmap="RdBu_r",
    norm=norm,
    shading="flat",
    edgecolors="#ffffff",
    linewidth=1.1,
    antialiased=True,
)

ax.set_xlim(0, len(FREQUENCY_ORDER))
ax.set_ylim(len(BENCHMARK_ORDER), 0)
ax.set_xticks(np.arange(len(FREQUENCY_ORDER)) + 0.5)
ax.set_xticklabels(FREQUENCY_ORDER)
ax.set_yticks(np.arange(len(BENCHMARK_ORDER)) + 0.5)
ax.set_yticklabels(BENCHMARK_ORDER)
ax.set_xlabel("Temporal resolution")
ax.set_title("Figure 5.1 — last_CV net PnL (`pnl_sum`) across all model-frequency configurations")
ax.grid(False)

# Force a stable rectangular cell geometry in saved output.
ax.set_box_aspect(len(BENCHMARK_ORDER) / (len(FREQUENCY_ORDER) * 1.7))

for i, model in enumerate(BENCHMARK_ORDER):
    for j, freq in enumerate(FREQUENCY_ORDER):
        value = float(heat.loc[model, freq])
        trades = int(trade_counts.loc[model, freq])
        color = "white" if abs(value) > 0.55 * vmax else THESIS_COLORS["ink"]
        ax.text(j + 0.5, i + 0.5, f"{value:.3f}\nn={trades}", ha="center", va="center", fontsize=8, color=color)

cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label("Net PnL (`pnl_sum`), centered at zero")
ax.text(
    0.0,
    -0.13,
    "last_CV only; annotations report net PnL and trade count. Color intensity is not an uncertainty interval or formal dominance test.",
    transform=ax.transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_1_path = save_thesis_figure(
    fig,
    figure_id="5.1",
    filename_stem="fig_5_1_benchmark_overview",
    title="Benchmark overview by frequency, graph family, and operator",
    source_artifacts=['final_runs/*/final_report.csv'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 5",
    validation_checks="All 18 last_CV rows shown; diverging scale centered at zero; no significance claim.",
)
fig_5_1_path


## Figure 5.2 — Cumulative gross PnL paths for representative final-holdout models

**Purpose in the thesis**  
This figure moves the benchmark interpretation from endpoint totals to realized path behaviour before transaction costs. The text uses it to show that positive gross signal exists even when later net outcomes differ.

**What the figure shows**  
The plot displays cumulative gross PnL paths for representative final-holdout models: the best 5min model, the best 1min model, and the least-negative 1sec model after costs. The ETH midpoint index is included only as market-context reference, not as a claim that the strategies follow price passively.

**Text interpretation**  
The figure supports the conclusion that the 1min model extracts more pre-cost opportunity than the 5min model, and that the 1sec representative can also produce positive gross PnL before costs.

**Metadata**
- Figure type: `empirical-result`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 5
- Primary inputs: final-holdout trade logs and ETH midpoint reference data
- Acceptance check: The three representative models are plotted over the common clock-time interval and the metric is gross PnL.


In [ ]:
import matplotlib.dates as mdates

# Figure 5.2/5.3 alignment policy:
# - PnL paths are computed from each model's full final-holdout trade log.
# - The cross-frequency figures display the strict common clock-time intersection
#   from 2021-04-17 03:25:34 UTC through 2021-04-18 05:05:00 UTC, derived from
#   `paper_artifacts/final_holdout_alignment_table.md`.
# - Legends report full-holdout totals; endpoint labels report displayed-window totals
#   when strict clipping excludes a tail trade.
# - ETH midpoint is read from the 1min source as a common market-context reference;
#   the very large 1sec source is intentionally avoided for these overview figures.

REPRESENTATIVE_TRADE_PATH_SPECS = [
    {
        "frequency": "5min",
        "model_label": "base-gnn-conv",
        "display_label": "5min base-gnn-conv",
        "family": "base-gnn",
        "color": FAMILY_COLORS["base-gnn"],
        "trade_log": REPO_ROOT / "final_runs/5min-base-gnn/adaptive_conv/adaptive_conv_final_holdout_last_cv_trade_log.csv",
        "holdout_start": pd.Timestamp("2021-04-17T03:25:00Z"),
        "holdout_end": pd.Timestamp("2021-04-18T05:05:00Z"),
        "expected_gross": 0.0281563144235406,
        "expected_net": 0.0203563144235406,
    },
    {
        "frequency": "1min",
        "model_label": "base-gnn-conv",
        "display_label": "1min base-gnn-conv",
        "family": "base-gnn",
        "color": THESIS_COLORS["teal"],
        "trade_log": REPO_ROOT / "final_runs/1min-base-gnn-conv/adaptive_conv/adaptive_conv_final_holdout_last_cv_trade_log.csv",
        "holdout_start": pd.Timestamp("2021-04-17T03:25:00Z"),
        "holdout_end": pd.Timestamp("2021-04-18T05:07:00Z"),
        "expected_gross": 0.0596942877309629,
        "expected_net": 0.0200942877309629,
    },
    {
        "frequency": "1sec",
        "model_label": "base-gnn-mpnn",
        "display_label": "1sec base-gnn-mpnn",
        "family": "base-gnn",
        "color": THESIS_COLORS["purple"],
        "trade_log": REPO_ROOT / "final_runs/1sec-base-gnn-mpnn/adaptive_mpnn/adaptive_mpnn_final_holdout_last_cv_trade_log.csv",
        "holdout_start": pd.Timestamp("2021-04-17T03:25:34Z"),
        "holdout_end": pd.Timestamp("2021-04-18T05:10:01Z"),
        "expected_gross": 0.0526793222416017,
        "expected_net": -0.0658206777583982,
    },
]

ONE_SEC_COST_DRAG_SPECS = [
    {
        "frequency": "1sec",
        "model_label": "memory-gnn-conv",
        "display_label": "1sec memory-gnn-conv",
        "family": "memory-gnn",
        "color": FAMILY_COLORS["memory-gnn"],
        "trade_log": REPO_ROOT / "final_runs/1sec-memory-gnn-conv/conv/conv_final_holdout_last_cv_trade_log.csv",
        "holdout_start": pd.Timestamp("2021-04-17T03:25:34Z"),
        "holdout_end": pd.Timestamp("2021-04-18T05:10:02Z"),
        "expected_gross": 0.4120322243152259,
        "expected_net": -1.163267775684774,
    },
    {
        "frequency": "1sec",
        "model_label": "base-gnn-mpnn",
        "display_label": "1sec base-gnn-mpnn",
        "family": "base-gnn",
        "color": THESIS_COLORS["purple"],
        "trade_log": REPO_ROOT / "final_runs/1sec-base-gnn-mpnn/adaptive_mpnn/adaptive_mpnn_final_holdout_last_cv_trade_log.csv",
        "holdout_start": pd.Timestamp("2021-04-17T03:25:34Z"),
        "holdout_end": pd.Timestamp("2021-04-18T05:10:01Z"),
        "expected_gross": 0.0526793222416017,
        "expected_net": -0.0658206777583982,
    },
]

ETH_PRICE_PATHS = {
    "5min": REPO_ROOT / "../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_5min.csv",
    "1min": REPO_ROOT / "../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1min.csv",
    "1sec": REPO_ROOT / "../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1sec.csv",
}

TRADE_COST = 0.0003
CROSS_FREQUENCY_DISPLAY_START = pd.Timestamp("2021-04-17T03:25:34Z")
CROSS_FREQUENCY_DISPLAY_END = pd.Timestamp("2021-04-18T05:05:00Z")
ONE_SEC_DISPLAY_START = pd.Timestamp("2021-04-17T03:25:34Z")
ONE_SEC_DISPLAY_END = pd.Timestamp("2021-04-18T05:10:02Z")  # includes the final memory-gnn-conv exit observed in the trade log

REQUIRED_TRADE_COLUMNS = {"entry_timestamp", "exit_timestamp", "gross_pnl", "net_pnl"}


def _read_trade_log(spec):
    path = spec["trade_log"]
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    missing = REQUIRED_TRADE_COLUMNS.difference(df.columns)
    if missing:
        raise ValueError(f"{path} missing required columns: {sorted(missing)}")
    df = df.copy()
    df["entry_timestamp"] = pd.to_datetime(df["entry_timestamp"], utc=True)
    df["exit_timestamp"] = pd.to_datetime(df["exit_timestamp"], utc=True)
    return df.sort_values("exit_timestamp").reset_index(drop=True)


def load_trade_path(spec, metric):
    if metric not in {"gross_pnl", "net_pnl"}:
        raise ValueError("metric must be 'gross_pnl' or 'net_pnl'")
    trades = _read_trade_log(spec)
    path = trades[["exit_timestamp", metric]].rename(columns={"exit_timestamp": "timestamp", metric: "pnl"})
    path[f"cum_{metric}"] = path["pnl"].cumsum()
    anchor = pd.DataFrame({"timestamp": [spec["holdout_start"]], "pnl": [0.0], f"cum_{metric}": [0.0]})
    path = pd.concat([anchor, path], ignore_index=True).sort_values("timestamp", kind="mergesort").reset_index(drop=True)
    full_total = float(trades[metric].sum())
    path["display_label"] = spec["display_label"]
    path["frequency"] = spec["frequency"]
    path["model_label"] = spec["model_label"]
    return path, full_total, int(len(trades))


def load_dual_metric_trade_path(spec):
    trades = _read_trade_log(spec)
    path = trades[["exit_timestamp", "gross_pnl", "net_pnl"]].rename(columns={"exit_timestamp": "timestamp"})
    path["cum_gross_pnl"] = path["gross_pnl"].cumsum()
    path["cum_net_pnl"] = path["net_pnl"].cumsum()
    path["cum_cost_drag"] = path["cum_gross_pnl"] - path["cum_net_pnl"]
    anchor = pd.DataFrame({
        "timestamp": [spec["holdout_start"]],
        "gross_pnl": [0.0],
        "net_pnl": [0.0],
        "cum_gross_pnl": [0.0],
        "cum_net_pnl": [0.0],
        "cum_cost_drag": [0.0],
    })
    path = pd.concat([anchor, path], ignore_index=True).sort_values("timestamp", kind="mergesort").reset_index(drop=True)
    gross_total = float(trades["gross_pnl"].sum())
    net_total = float(trades["net_pnl"].sum())
    final_cost_drag = gross_total - net_total
    if not np.isclose(final_cost_drag, len(trades) * TRADE_COST, atol=1e-9):
        raise AssertionError(f"Cost drag mismatch for {spec['display_label']}: {final_cost_drag} vs {len(trades) * TRADE_COST}")
    return path, gross_total, net_total, int(len(trades)), final_cost_drag


def path_on_window(path, value_col, start, end):
    path = path.sort_values("timestamp")
    prior_start = path[path["timestamp"] <= start]
    prior_end = path[path["timestamp"] <= end]
    start_value = float(prior_start[value_col].iloc[-1]) if len(prior_start) else 0.0
    end_value = float(prior_end[value_col].iloc[-1]) if len(prior_end) else start_value
    clipped = path[(path["timestamp"] >= start) & (path["timestamp"] <= end)][["timestamp", value_col]].copy()
    boundary = pd.DataFrame({"timestamp": [start, end], value_col: [start_value, end_value]})
    clipped = pd.concat([boundary, clipped], ignore_index=True).sort_values("timestamp", kind="mergesort")
    clipped = clipped.drop_duplicates(subset=["timestamp"], keep="last").reset_index(drop=True)
    return clipped, start_value, end_value


def load_eth_midpoint_reference(start=CROSS_FREQUENCY_DISPLAY_START, end=CROSS_FREQUENCY_DISPLAY_END, frequency="1min"):
    path = ETH_PRICE_PATHS[frequency]
    if not path.exists():
        raise FileNotFoundError(path)
    eth = pd.read_csv(path, usecols=["system_time", "midpoint"])
    missing = {"system_time", "midpoint"}.difference(eth.columns)
    if missing:
        raise ValueError(f"{path} missing required columns: {sorted(missing)}")
    eth["system_time"] = pd.to_datetime(eth["system_time"], utc=True)
    eth = eth[(eth["system_time"] >= start) & (eth["system_time"] <= end)].copy()
    eth = eth.sort_values("system_time").reset_index(drop=True)
    if eth.empty:
        raise ValueError(f"No ETH midpoint rows in {path} for {start} to {end}")
    eth["midpoint_index"] = eth["midpoint"] / eth["midpoint"].iloc[0] * 100.0
    return eth


def annotate_endpoint(ax, x, y, text, color, dy=0.0):
    ax.annotate(
        text,
        xy=(x, y),
        xytext=(6, dy),
        textcoords="offset points",
        ha="left",
        va="center",
        fontsize=8,
        color=color,
        bbox={"boxstyle": "round,pad=0.18", "fc": "white", "ec": color, "alpha": 0.78, "lw": 0.6},
    )

def align_secondary_axis_value(ax_left, ax_right, left_value=0.0, right_value=100.0, right_data=None):
    """Align a right-axis reference value with a left-axis reference value.

    Used for Figures 5.2/5.3 so ETH midpoint index 100 sits exactly on the
    same horizontal gridline as cumulative PnL 0.00.
    """
    left_min, left_max = ax_left.get_ylim()
    if not (left_min < left_value < left_max):
        margin = max((left_max - left_min) * 0.05, 1e-6)
        left_min = min(left_min, left_value - margin)
        left_max = max(left_max, left_value + margin)
        ax_left.set_ylim(left_min, left_max)
    frac = (left_value - left_min) / (left_max - left_min)
    frac = min(max(float(frac), 1e-6), 1.0 - 1e-6)

    if right_data is None:
        right_min, right_max = ax_right.get_ylim()
    else:
        values = pd.Series(right_data).dropna().astype(float)
        right_min, right_max = float(values.min()), float(values.max())
    lower_span = (right_value - right_min) / frac if right_min < right_value else 0.0
    upper_span = (right_max - right_value) / (1.0 - frac) if right_max > right_value else 0.0
    total_span = max(lower_span, upper_span, 1e-6)
    ax_right.set_ylim(right_value - frac * total_span, right_value + (1.0 - frac) * total_span)



In [ ]:
eth_reference = load_eth_midpoint_reference()
trade_path_cache = {}
for spec in REPRESENTATIVE_TRADE_PATH_SPECS:
    for metric in ["gross_pnl", "net_pnl"]:
        trade_path_cache[(spec["display_label"], metric)] = load_trade_path(spec, metric)

fig, ax = plt.subplots(figsize=(12.5, 5.8), constrained_layout=True)
ax2 = ax.twinx()

for spec in REPRESENTATIVE_TRADE_PATH_SPECS:
    path, full_total, n_trades = trade_path_cache[(spec["display_label"], "gross_pnl")]
    clipped, _, shown_end = path_on_window(path, "cum_gross_pnl", CROSS_FREQUENCY_DISPLAY_START, CROSS_FREQUENCY_DISPLAY_END)
    label = f"{spec['display_label']} (full {full_total:.3f}, n={n_trades})"
    ax.step(clipped["timestamp"], clipped["cum_gross_pnl"], where="post", linewidth=2.0, color=spec["color"], label=label)
    annotate_endpoint(ax, clipped["timestamp"].iloc[-1], shown_end, f"shown {shown_end:.3f}", spec["color"])

ax2.plot(
    eth_reference["system_time"],
    eth_reference["midpoint_index"],
    color=THESIS_COLORS["slate"],
    linewidth=1.5,
    alpha=0.62,
    label="ETH midpoint (indexed, right axis)",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xlim(CROSS_FREQUENCY_DISPLAY_START, CROSS_FREQUENCY_DISPLAY_END)
ax.set_ylabel("Cumulative gross PnL")
ax2.set_ylabel("ETH midpoint index (start = 100)", color=THESIS_COLORS["slate"])
ax2.tick_params(axis="y", labelcolor=THESIS_COLORS["slate"])
align_secondary_axis_value(ax, ax2, left_value=0.0, right_value=100.0, right_data=eth_reference["midpoint_index"])
ax.set_title("Figure 5.2 — Cumulative gross PnL paths for representative final-holdout models")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
handles_1, labels_1 = ax.get_legend_handles_labels()
handles_2, labels_2 = ax2.get_legend_handles_labels()
ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left", frameon=False)
ax.text(
    0.01,
    -0.18,
    "PnL paths are event-based and update at realized trade exits. Display window is the strict cross-frequency intersection; legend totals use full final-holdout trade logs.",
    transform=ax.transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_2_path = save_thesis_figure(
    fig,
    figure_id="5.2",
    filename_stem="fig_5_2_trade_paths_gross_pnl",
    title="Cumulative gross PnL paths for representative final-holdout models",
    source_artifacts=['*_final_holdout_last_cv_trade_log.csv', '../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1min.csv'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 5",
    validation_checks="Path totals validated against expected artifact values; event-exit path note included.",
)
fig_5_2_path


## Figure 5.3 — Cumulative net PnL paths for representative final-holdout models

**Purpose in the thesis**  
This figure repeats the representative path comparison after transaction costs. The text uses it as the deployment-relevant counterpart to Figure 5.2.

**What the figure shows**  
The plot displays cumulative net PnL paths for the same representative final-holdout models and the same ETH midpoint context. It keeps the model selection and time alignment fixed so the only interpretive change is gross-to-net cost adjustment.

**Text interpretation**  
The figure supports the central cost-aware conclusion: the 1min model finds more gross opportunity but gives up more to turnover costs, while the 1sec representative's positive gross signal does not survive the transaction-cost proxy.

**Metadata**
- Figure type: `empirical-result`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 5
- Primary inputs: final-holdout trade logs and ETH midpoint reference data
- Acceptance check: The same representative models as Figure 5.2 are plotted and the metric is post-cost net PnL.


In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.8), constrained_layout=True)
ax2 = ax.twinx()

for spec in REPRESENTATIVE_TRADE_PATH_SPECS:
    path, full_total, n_trades = trade_path_cache[(spec["display_label"], "net_pnl")]
    clipped, _, shown_end = path_on_window(path, "cum_net_pnl", CROSS_FREQUENCY_DISPLAY_START, CROSS_FREQUENCY_DISPLAY_END)
    label = f"{spec['display_label']} (full {full_total:.3f}, n={n_trades})"
    ax.step(clipped["timestamp"], clipped["cum_net_pnl"], where="post", linewidth=2.0, color=spec["color"], label=label)
    annotate_endpoint(ax, clipped["timestamp"].iloc[-1], shown_end, f"shown {shown_end:.3f}", spec["color"])

ax2.plot(
    eth_reference["system_time"],
    eth_reference["midpoint_index"],
    color=THESIS_COLORS["slate"],
    linewidth=1.5,
    alpha=0.62,
    label="ETH midpoint (indexed, right axis)",
)
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xlim(CROSS_FREQUENCY_DISPLAY_START, CROSS_FREQUENCY_DISPLAY_END)
ax.set_ylabel("Cumulative net PnL (`pnl_sum` path)")
ax2.set_ylabel("ETH midpoint index (start = 100)", color=THESIS_COLORS["slate"])
ax2.tick_params(axis="y", labelcolor=THESIS_COLORS["slate"])
align_secondary_axis_value(ax, ax2, left_value=0.0, right_value=100.0, right_data=eth_reference["midpoint_index"])
ax.set_title("Figure 5.3 — Cumulative net PnL paths for representative final-holdout models")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
handles_1, labels_1 = ax.get_legend_handles_labels()
handles_2, labels_2 = ax2.get_legend_handles_labels()
ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc="lower left", frameon=False)
ax.text(
    0.01,
    -0.18,
    "Net PnL is the deployment-oriented path. The 1min line gives back more of its gross signal to costs than the 5min line despite similar full-holdout totals.",
    transform=ax.transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_3_path = save_thesis_figure(
    fig,
    figure_id="5.3",
    filename_stem="fig_5_3_trade_paths_net_pnl",
    title="Cumulative net PnL paths for representative final-holdout models",
    source_artifacts=['*_final_holdout_last_cv_trade_log.csv', '../Graph_Neural_Network_for_Market_Microstructure/dataset/ETH_1min.csv'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 5",
    validation_checks="Net totals validated against expected artifact values; cost interpretation note included.",
)
fig_5_3_path


## Figure 5.4 — Gross versus net PnL for 1sec models

**Purpose in the thesis**  
This figure isolates the high-frequency regime where the difference between signal extraction and deployability is strongest. The text uses it to explain why the one-second experiments are informative even though all net PnL values are negative.

**What the figure shows**  
The plot compares gross PnL, net PnL, and cost drag for the six 1sec configurations, with trade counts shown as part of the interpretation.

**Text interpretation**  
The figure supports the conclusion that memory-based 1sec models can extract large gross signal, but their high turnover causes transaction costs to dominate the net result.

**Metadata**
- Figure type: `empirical-result`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 5
- Primary inputs: `final_runs/1sec-*/final_report.csv`
- Acceptance check: Gross PnL, net PnL, cost drag, and trade counts are visible for all six 1sec models.


In [ ]:
one_sec_df = benchmark_df[benchmark_df["frequency"] == "1sec"].copy().sort_values("order").reset_index(drop=True)
one_sec_df["x_label"] = one_sec_df["model_label"].map({
    "base-gnn-conv": "Base\nConv",
    "base-gnn-mpnn": "Base\nMPNN",
    "multi-gnn-conv": "Multi\nConv",
    "multi-gnn-mpnn": "Multi\nMPNN",
    "memory-gnn-conv": "Memory\nConv",
    "memory-gnn-mpnn": "Memory\nMPNN",
})
one_sec_df["cost_drag"] = one_sec_df["gross_pnl_sum"] - one_sec_df["pnl_sum"]
one_sec_df["implied_cost_drag"] = one_sec_df["n_trades"] * TRADE_COST
assert np.allclose(one_sec_df["cost_drag"], one_sec_df["implied_cost_drag"], atol=1e-8)
one_sec_df[["model_label", "gross_pnl_sum", "pnl_sum", "cost_drag", "n_trades", "trade_rate"]]


In [ ]:
fig, ax = plt.subplots(figsize=(11.5, 5.2), constrained_layout=True)
x = np.arange(len(one_sec_df))
width = 0.36

ax.bar(x - width / 2, one_sec_df["gross_pnl_sum"], width=width, color="#cbd5e1", edgecolor="black", linewidth=0.5, label="gross_pnl_sum")
ax.bar(x + width / 2, one_sec_df["pnl_sum"], width=width, color=one_sec_df["family"].map(FAMILY_COLORS), edgecolor="black", linewidth=0.5, label="pnl_sum")
ax.axhline(0, color="black", linewidth=0.9)
ax.set_xticks(x)
ax.set_xticklabels(one_sec_df["x_label"])
ax.set_ylabel("PnL")
ax.set_title("Figure 5.2 — Gross versus net PnL at 1sec")

for x_i, gross_value, net_value in zip(x, one_sec_df["gross_pnl_sum"], one_sec_df["pnl_sum"]):
    ax.text(x_i - width / 2, gross_value + 0.02, f"{gross_value:.3f}", ha="center", va="bottom", fontsize=8, rotation=90)
    ax.text(x_i + width / 2, net_value - 0.05 if net_value < 0 else net_value + 0.02, f"{net_value:.3f}", ha="center", va="top" if net_value < 0 else "bottom", fontsize=8, rotation=90)

ax2 = ax.twinx()
ax2.plot(x, one_sec_df["n_trades"], color="#7c2d12", marker="o", linewidth=1.8, label="n_trades")
ax2.set_ylabel("Number of trades", color="#7c2d12")
ax2.tick_params(axis="y", labelcolor="#7c2d12")
for x_i, trades in zip(x, one_sec_df["n_trades"]):
    ax2.text(x_i, trades + max(one_sec_df["n_trades"]) * 0.03, f"{int(trades)}", ha="center", va="bottom", fontsize=8, color="#7c2d12")

handles_1, labels_1 = ax.get_legend_handles_labels()
handles_2, labels_2 = ax2.get_legend_handles_labels()
ax.legend(handles_1 + handles_2, labels_1 + labels_2, loc="upper left", frameon=False)
#ax.text(0.98, 0.96, "memory-gnn-conv: strongest gross signal but\nturnover overwhelms post-cost viability", transform=ax.transAxes, ha="right", va="top", fontsize=9, color="#7c2d12")

mem = one_sec_df[one_sec_df["model_label"] == "memory-gnn-conv"].iloc[0]
ax.annotate(
    f"memory-gnn-conv\ngross {mem['gross_pnl_sum']:.3f}\nnet {mem['pnl_sum']:.3f}",
    xy=(int(mem["order"]) + width / 2, mem["pnl_sum"]),
    xytext=(3.8, -0.62),
    arrowprops={"arrowstyle": "->", "color": "#7c2d12", "lw": 1.0},
    bbox={"boxstyle": "round,pad=0.25", "fc": "white", "ec": "#7c2d12", "alpha": 0.9},
    fontsize=8.5,
    color="#7c2d12",
)
fig


In [ ]:
# Figure 5.4 is rendered in the previous cell; this cell preserves the prepared plotting table for inspection.
one_sec_df[["model_label", "gross_pnl_sum", "pnl_sum", "cost_drag", "n_trades"]]


In [ ]:
fig_5_4_path = save_thesis_figure(
    fig,
    figure_id="5.4",
    filename_stem="fig_5_4_gross_vs_net_1sec",
    title="Gross versus net PnL at 1sec",
    source_artifacts=['final_runs/1sec-*/final_report.csv'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 5",
    validation_checks="Exactly six 1sec models; cost drag equals n_trades × 0.0003.",
)
fig_5_4_path


## Figure 5.5 — One-second cumulative gross-versus-net PnL paths for memory and baseline models

**Purpose in the thesis**  
This figure makes the one-second cost-drag mechanism concrete at path level. The text uses it to show how a model with positive pre-cost signal can become strongly negative after costs.

**What the figure shows**  
The plot overlays cumulative gross and net PnL paths for `memory-gnn-conv` and `base-gnn-mpnn` in the 1sec final holdout. Solid lines represent gross PnL and dashed lines represent net PnL under the shared cost rule.

**Text interpretation**  
The figure supports the conclusion that `memory-gnn-conv` is not simply failing to find high-frequency structure; it finds gross signal but expresses it through 5251 trades, producing much larger cost drag than the lower-turnover baseline.

**Metadata**
- Figure type: `empirical-result`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 5
- Primary inputs: 1sec final-holdout trade logs for `memory-gnn-conv` and `base-gnn-mpnn`
- Acceptance check: Gross/net paths are shown for both models and the final cost-drag interpretation matches trade counts.


In [ ]:
fig, ax = plt.subplots(figsize=(12.8, 6.4), constrained_layout=True)

cost_drag_cache = {}
line_handles = []
annotation_lines = []
for spec in ONE_SEC_COST_DRAG_SPECS:
    path, gross_total, net_total, n_trades, final_cost_drag = load_dual_metric_trade_path(spec)
    cost_drag_cache[spec["display_label"]] = (path, gross_total, net_total, n_trades, final_cost_drag)
    clipped_gross, _, shown_gross = path_on_window(path, "cum_gross_pnl", ONE_SEC_DISPLAY_START, ONE_SEC_DISPLAY_END)
    clipped_net, _, shown_net = path_on_window(path, "cum_net_pnl", ONE_SEC_DISPLAY_START, ONE_SEC_DISPLAY_END)
    merged = pd.merge_asof(
        clipped_gross.rename(columns={"cum_gross_pnl": "gross"}).sort_values("timestamp"),
        clipped_net.rename(columns={"cum_net_pnl": "net"}).sort_values("timestamp"),
        on="timestamp",
        direction="nearest",
        tolerance=pd.Timedelta(seconds=0),
    )
    if merged["net"].isna().any() or len(merged) != len(clipped_gross):
        merged = pd.DataFrame({
            "timestamp": clipped_gross["timestamp"],
            "gross": clipped_gross["cum_gross_pnl"].to_numpy(),
            "net": np.interp(
                mdates.date2num(clipped_gross["timestamp"]),
                mdates.date2num(clipped_net["timestamp"]),
                clipped_net["cum_net_pnl"],
            ),
        })
    gross_label = f"{spec['display_label']} gross"
    net_label = f"{spec['display_label']} net"
    gross_line = ax.step(
        clipped_gross["timestamp"],
        clipped_gross["cum_gross_pnl"],
        where="post",
        color=spec["color"],
        linewidth=2.1,
        linestyle="-",
        label=gross_label,
        zorder=3,
    )[0]
    net_line = ax.step(
        clipped_net["timestamp"],
        clipped_net["cum_net_pnl"],
        where="post",
        color=spec["color"],
        linewidth=2.1,
        linestyle="--",
        label=net_label,
        zorder=3,
    )[0]
    line_handles.extend([gross_line, net_line])
    ax.fill_between(
        merged["timestamp"],
        merged["gross"],
        merged["net"],
        step="post",
        color=spec["color"],
        alpha=0.10 if spec["model_label"] == "memory-gnn-conv" else 0.16,
        zorder=1,
    )
    annotation_lines.append(
        f"{spec['display_label']}: gross {gross_total:.3f}, net {net_total:.3f}, "
        f"trades {n_trades}, drag {final_cost_drag:.4f}"
    )

ax.axhline(0, color="black", linewidth=0.9)
ax.set_xlim(ONE_SEC_DISPLAY_START, ONE_SEC_DISPLAY_END)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
ax.set_xlabel("Final-holdout UTC clock time")
ax.set_ylabel("Cumulative PnL")
ax.set_title("Figure 5.5 — One-second cumulative gross-versus-net PnL paths")
ax.legend(handles=line_handles, loc="lower left", frameon=False, ncols=2)
ax.text(
    0.985,
    0.04,
    "\n".join(annotation_lines),
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=8.3,
    bbox={"boxstyle": "round,pad=0.35", "fc": "white", "ec": "#cbd5e1", "alpha": 0.9},
)
ax.text(
    0.01,
    -0.18,
    "Single-axis view: solid lines are gross PnL, dashed lines are net PnL; shaded regions show each model's transaction-cost drag. The memory model's scale necessarily compresses the lower-turnover baseline.",
    transform=ax.transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_5_path = save_thesis_figure(
    fig,
    figure_id="5.5",
    filename_stem="fig_5_5_1sec_memory_cost_drag_paths",
    title="One-second memory cost-drag path comparison",
    source_artifacts=['1sec-memory-gnn-conv trade log', '1sec-base-gnn-mpnn trade log'],
    rendering_method="Python, artifact-grounded",
    main_text_section="Chapter 5",
    validation_checks="Gross/net paths and shaded cost drag are built from trade logs.",
)
fig_5_5_path


In [ ]:
validation_rows = []
for spec in REPRESENTATIVE_TRADE_PATH_SPECS + [ONE_SEC_COST_DRAG_SPECS[0]]:
    path, gross_total, net_total, n_trades, final_cost_drag = load_dual_metric_trade_path(spec)
    expected_gross = spec["expected_gross"]
    expected_net = spec["expected_net"]
    assert abs(gross_total - expected_gross) <= 1e-9, (spec["display_label"], gross_total, expected_gross)
    assert abs(net_total - expected_net) <= 1e-9, (spec["display_label"], net_total, expected_net)
    assert abs(final_cost_drag - (gross_total - net_total)) <= 1e-12
    assert np.isclose(final_cost_drag, n_trades * TRADE_COST, atol=1e-9)
    validation_rows.append({
        "frequency": spec["frequency"],
        "model": spec["model_label"],
        "n_trades": n_trades,
        "gross_total": gross_total,
        "net_total": net_total,
        "cost_drag": final_cost_drag,
        "first_trade": path["timestamp"].iloc[1] if len(path) > 1 else pd.NaT,
        "last_trade": path["timestamp"].iloc[-1],
    })

for output_path in [fig_5_2_path, fig_5_3_path, fig_5_5_path]:
    assert output_path.exists() and output_path.stat().st_size > 0, output_path

eth_check = load_eth_midpoint_reference()
assert {"system_time", "midpoint"}.issubset(eth_check.columns)

trade_path_validation_summary = pd.DataFrame(validation_rows)
trade_path_validation_summary


## Figure 5.6 — `last_CV` versus `final_refit` as deployment-oriented model states

**Purpose in the thesis**  
This figure summarizes the model-state comparison before the selected numerical cases in Sections 5.6.1–5.6.4. The current thesis version uses option 2 as the main Figure 5.6.

**What the figure shows**  
The plot places `last_CV` and `final_refit` for selected models in final-holdout `trade_auc` × `dir_auc` diagnostic space. Circles mark `last_CV`, squares mark `final_refit`, arrows show the state transition, and marker fill encodes whether post-cost `pnl_sum` is positive or negative.

**Text interpretation**  
The figure supports the conclusion that `last_CV` and `final_refit` are related but not interchangeable. Refit can improve AUC diagnostics without guaranteeing a better net PnL outcome, so `last_CV` remains the primary deployment-oriented reference.

**Metadata**
- Figure type: `empirical-diagnostic`
- Rendering method: `executable artifact-grounded plot`
- Main text section: Chapter 5
- Primary inputs: `*_final_holdout_model_comparison_summary.csv`
- Acceptance check: Eight final-holdout points are shown, arrows connect selected model states, and `last_CV` is visually distinct from `final_refit`.


In [ ]:
state_case_files = {
    "5min | base-gnn-conv": REPO_ROOT / "final_runs/5min-base-gnn/adaptive_conv/adaptive_conv_final_holdout_model_comparison_summary.csv",
    "1min | base-gnn-conv": REPO_ROOT / "final_runs/1min-base-gnn-conv/adaptive_conv/adaptive_conv_final_holdout_model_comparison_summary.csv",
    "1sec | memory-gnn-conv": REPO_ROOT / "final_runs/1sec-memory-gnn-conv/conv/conv_final_holdout_model_comparison_summary.csv",
    "5min | multi-gnn-conv": REPO_ROOT / "final_runs/5min-multi-gnn/dynamic_rel_conv/dynamic_rel_conv_final_holdout_model_comparison_summary.csv",
}

state_rows = []
for case_label, csv_path in state_case_files.items():
    df = pd.read_csv(csv_path)
    df = df[df["model_role"].isin(["last_cv_fold_model", "final_refit_model"])].copy()
    df["case_label"] = case_label
    df["state"] = df["model_role"].map({"last_cv_fold_model": "last_CV", "final_refit_model": "final_refit"})
    state_rows.append(df[["case_label", "state", "pnl_sum", "dir_auc", "trade_auc", "n_trades", "gross_pnl_sum"]])

state_comparison_df = pd.concat(state_rows, ignore_index=True)
state_case_order = list(state_case_files.keys())
state_comparison_df["case_order"] = state_comparison_df["case_label"].map({label: i for i, label in enumerate(state_case_order)})
state_comparison_df["state_order"] = state_comparison_df["state"].map({"last_CV": 0, "final_refit": 1})
state_comparison_df.sort_values(["case_order", "state_order"])


In [ ]:
state_plot_df = state_comparison_df.sort_values(["case_order", "state_order"]).reset_index(drop=True)
state_colors = {"last_CV": "#4c78a8", "final_refit": "#f58518"}
state_y = np.arange(len(state_case_order))[::-1]


In [ ]:
state_marker = {"last_CV": "o", "final_refit": "s"}
state_label = {"last_CV": "last_CV (circle)", "final_refit": "final_refit (square)"}
case_arrow_colors = {
    "5min | base-gnn-conv": FAMILY_COLORS["base-gnn"],
    "1min | base-gnn-conv": FAMILY_COLORS["base-gnn"],
    "1sec | memory-gnn-conv": FAMILY_COLORS["memory-gnn"],
    "5min | multi-gnn-conv": FAMILY_COLORS["multi-gnn"],
}
case_text_offsets = {
    "5min | base-gnn-conv": (7, 8),
    "1min | base-gnn-conv": (7, -18),
    "1sec | memory-gnn-conv": (7, -18),
    "5min | multi-gnn-conv": (7, 8),
}

pnl_values = state_plot_df["pnl_sum"].astype(float).to_numpy()
pos_max = float(np.max(pnl_values[pnl_values > 0])) if np.any(pnl_values > 0) else 0.0
neg_max = float(np.max(np.abs(pnl_values[pnl_values < 0]))) if np.any(pnl_values < 0) else 0.0


def state_pnl_color(value):
    """Encode post-cost final-holdout PnL with thesis green/red fills."""
    value = float(value)
    if value >= 0:
        scale = min(value / pos_max, 1.0) if pos_max > 0 else 0.0
        return mpl.colors.to_rgba(THESIS_COLORS["green"], 0.42 + 0.48 * scale)
    scale = min(abs(value) / neg_max, 1.0) if neg_max > 0 else 0.0
    return mpl.colors.to_rgba(THESIS_COLORS["red"], 0.38 + 0.50 * scale)


x_values = state_plot_df["trade_auc"].astype(float)
y_values = state_plot_df["dir_auc"].astype(float)
x_pad = max((float(x_values.max()) - float(x_values.min())) * 0.12, 0.025)
y_pad = max((float(y_values.max()) - float(y_values.min())) * 0.18, 0.025)
xlim = (max(0.45, float(x_values.min()) - x_pad), min(0.90, float(x_values.max()) + x_pad))
ylim = (max(0.48, float(y_values.min()) - y_pad), min(0.75, float(y_values.max()) + y_pad))
#xlim = (0.45, 0.90)
#ylim = (0.45, 0.90)

fig, ax = plt.subplots(figsize=(12.8, 6.2), constrained_layout=False)
fig.subplots_adjust(left=0.085, right=0.985, top=0.86, bottom=0.20)
# Force a landscape plotting area in saved outputs. Without this, bbox_inches="tight"
# can make the single-panel option look like a narrow/square inset in the thesis.
ax.set_box_aspect(0.55)
ax.axvline(0.5, color=THESIS_COLORS["slate"], linewidth=1.0, linestyle="--", alpha=0.75)
ax.axhline(0.5, color=THESIS_COLORS["slate"], linewidth=1.0, linestyle="--", alpha=0.75)
diag_min = max(xlim[0], ylim[0])
diag_max = min(xlim[1], ylim[1])
ax.plot([diag_min, diag_max], [diag_min, diag_max], color=THESIS_COLORS["slate"], linewidth=0.9, linestyle=":", alpha=0.65)

for case in state_case_order:
    subset = state_plot_df[state_plot_df["case_label"] == case].sort_values("state_order")
    last_row = subset[subset["state"] == "last_CV"].iloc[0]
    refit_row = subset[subset["state"] == "final_refit"].iloc[0]
    arrow_color = case_arrow_colors.get(case, THESIS_COLORS["slate"])
    ax.annotate(
        "",
        xy=(float(refit_row["trade_auc"]), float(refit_row["dir_auc"])),
        xytext=(float(last_row["trade_auc"]), float(last_row["dir_auc"])),
        arrowprops=dict(arrowstyle="->", lw=1.7, color=arrow_color, alpha=0.86, shrinkA=7, shrinkB=7),
        zorder=2,
    )
    mid_x = (float(last_row["trade_auc"]) + float(refit_row["trade_auc"])) / 2
    mid_y = (float(last_row["dir_auc"]) + float(refit_row["dir_auc"])) / 2
    ax.annotate(
        case.replace(" | ", "\n"),
        xy=(mid_x+0.02, mid_y),
        #xytext=case_text_offsets.get(case, (-17, 7)),
        #textcoords="offset points",
        fontsize=8,
        color=THESIS_COLORS["ink"],
        ha="left",
        va="center",
    )

for _, row in state_plot_df.iterrows():
    ax.scatter(
        float(row["trade_auc"]),
        float(row["dir_auc"]),
        marker=state_marker[row["state"]],
        s=105,
        facecolor=state_pnl_color(row["pnl_sum"]),
        edgecolor="black",
        linewidth=0.55,
        zorder=4,
    )
    ax.annotate(
        f"{float(row['pnl_sum']):+.3f}",
        xy=(float(row["trade_auc"]), float(row["dir_auc"])),
        xytext=(-15, 10 if row["state"] == "final_refit" else -13),
        textcoords="offset points",
        fontsize=7.5,
        color="#334155",
        ha="left",
        va="center",
    )

ax.set_xlim(*xlim)
ax.set_ylim(*ylim)
ax.set_xlabel("Final-holdout trade AUC (`trade_auc`)")
ax.set_ylabel("Final-holdout directional AUC (`dir_auc`)")
ax.set_title("Figure 5.6 — Final-holdout diagnostic transition from `last_CV` to `final_refit`", fontsize=12, pad=12)
ax.grid(True, alpha=0.22)

state_handles = [
    plt.Line2D([0], [0], marker=marker, linestyle="None", markerfacecolor="white", markeredgecolor="black", markersize=8, label=state_label[state])
    for state, marker in state_marker.items()
]
pnl_handles = [
    plt.Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=THESIS_COLORS["green"], markeredgecolor="black", markersize=8, label="positive `pnl_sum`"),
    plt.Line2D([0], [0], marker="o", linestyle="None", markerfacecolor=THESIS_COLORS["red"], markeredgecolor="black", markersize=8, label="negative `pnl_sum`"),
]
arrow_handles = [
    plt.Line2D([0], [0], color=FAMILY_COLORS["base-gnn"], lw=1.7, label="base-gnn transition"),
    plt.Line2D([0], [0], color=FAMILY_COLORS["multi-gnn"], lw=1.7, label="multi-gnn transition"),
    plt.Line2D([0], [0], color=FAMILY_COLORS["memory-gnn"], lw=1.7, label="memory-gnn transition"),
]
ax.legend(handles=state_handles + pnl_handles + arrow_handles, frameon=False, loc="lower right", fontsize=8, ncols=1)
ax.text(
    0.0,
    -0.18,
    "Both markers use final-holdout/test metrics. Arrows point from deployment-primary `last_CV` to diagnostic `final_refit`; fill color reports post-cost net PnL.",
    transform=ax.transAxes,
    fontsize=8,
    color="#475569",
)
fig


In [ ]:
fig_5_6_path = save_thesis_figure(
    fig,
    figure_id="5.6",
    filename_stem="fig_5_6_option_2_auc_state_arrows",
    title="last_CV versus final_refit diagnostic transitions",
    source_artifacts=['*_final_holdout_model_comparison_summary.csv'],
    rendering_method="Python, artifact-grounded AUC transition scatter",
    main_text_section="Chapter 5",
    validation_checks="Eight final-holdout points are shown; circles mark last_CV, squares mark final_refit, and four arrows connect selected model states.",
)
fig_5_6_path


## Figure 6.1 — Deployment interpretation from prediction to post-cost evidence

**Purpose in the thesis**  
This concluding figure summarizes the practical interpretation chain. The text uses it to clarify why predictive diagnostics alone are not enough for deployment-oriented evidence.

**What the figure shows**  
The diagram links prediction quality, gross signal extraction, trade selectivity and turnover, transaction-cost adjustment, post-cost evidence, and model-state stability.

**Text interpretation**  
The figure supports the conclusion that a model is deployment-informative only after ranking quality, gross PnL, turnover, transaction costs, and model-state stability are interpreted together.

**Metadata**
- Figure type: `conceptual-summary`
- Rendering method: `executable vector diagram`
- Main text section: Chapter 6
- Primary inputs: `paper_artifacts/good_Paper_v1.md`, benchmark metric names from `final_runs/*/final_report.csv`
- Acceptance check: The chain runs from prediction diagnostics to post-cost evidence and includes model-state stability.


In [ ]:
fig, ax = plt.subplots(figsize=(9.2, 7.6), constrained_layout=True)
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

levels = [
    ("Prediction quality", "`dir_auc`   `trade_auc`   `rmse`", THESIS_COLORS["light_blue"], THESIS_COLORS["navy"]),
    ("Gross signal extraction", "`gross_pnl_sum`", THESIS_COLORS["light_teal"], THESIS_COLORS["teal"]),
    ("Trade selectivity and turnover", "`pnl_per_trade`   `n_trades`   `trade_rate`", THESIS_COLORS["light_orange"], THESIS_COLORS["orange"]),
    ("Transaction-cost adjustment", "shared cost proxy -> post-cost result", "#fef3c7", "#b45309"),
    ("Deployment-oriented evidence", "`pnl_sum` / `net_pnl` + model-state stability", "#fee2e2", THESIS_COLORS["red"]),
]
y_positions = [0.78, 0.61, 0.44, 0.27, 0.10]
for (title, body, face, edge), y in zip(levels, y_positions):
    thesis_box(ax, (0.16, y), 0.68, 0.105, title, body, facecolor=face, edgecolor=edge, title_color=edge, fontsize=9.5, title_size=10.5)

for y0, y1 in zip(y_positions[:-1], y_positions[1:]):
    thesis_arrow(ax, (0.5, y0), (0.5, y1 + 0.105), color=THESIS_COLORS["slate"], lw=1.7)

thesis_box(
    ax,
    (0.08, 0.015),
    0.84,
    0.06,
    "Interpretation rule",
    "Predictive metrics become thesis evidence only when read through gross signal, turnover, transaction costs, and `last_CV` / `final_refit` stability.",
    facecolor="white",
    edgecolor=THESIS_COLORS["slate"],
    title_color=THESIS_COLORS["ink"],
    fontsize=8.2,
    title_size=9.2,
)
ax.set_title("Figure 6.1 — Deployment interpretation from prediction to post-cost evidence", fontsize=13, pad=12)
fig


In [ ]:
fig_6_1_path = save_thesis_figure(
    fig,
    figure_id="6.1",
    filename_stem="fig_6_1_deployment_interpretation_chain",
    title="Deployment interpretation from prediction to post-cost evidence",
    source_artifacts=["paper_artifacts/good_Paper_v1.md", "final_runs/*/final_report.csv"],
    rendering_method="Python vector",
    main_text_section="Chapter 6",
    validation_checks="Includes predictive diagnostics, gross signal, turnover/selectivity, transaction-cost adjustment, post-cost PnL, and model-state stability.",
)
fig_6_1_path


## Final validation and figure manifest

This cell validates the executable notebook outputs, re-checks benchmark cost consistency, and writes the machine-readable and markdown manifest for generated figures. Manual/prompt figures documented above are produced in Miro.com and are therefore not required as notebook-generated files.


In [ ]:
required_outputs = [
    "fig_3_2_split_design_v2.png",
    "fig_3_3_triple_barrier_eth_midpoint.png",
    "fig_3_5_walkforward_model_states.png",
    "fig_5_1_benchmark_overview.png",
    "fig_5_2_trade_paths_gross_pnl.png",
    "fig_5_3_trade_paths_net_pnl.png",
    "fig_5_4_gross_vs_net_1sec.png",
    "fig_5_5_1sec_memory_cost_drag_paths.png",
    "fig_5_6_option_2_auc_state_arrows.png",
    "fig_6_1_deployment_interpretation_chain.png",
]
for name in required_outputs:
    path = FIGURES_DIR / name
    assert path.exists() and path.stat().st_size > 0, path

cost_check_df = load_benchmark_metrics(model_state="last_cv")
assert np.allclose(cost_check_df["gross_pnl_sum"] - cost_check_df["pnl_sum"], cost_check_df["n_trades"] * TRADE_COST, atol=1e-8)
assert len(cost_check_df) == 18
manifest_df, manifest_csv_path, manifest_md_path = write_manifest()
assert manifest_csv_path.exists() and manifest_csv_path.stat().st_size > 0
assert manifest_md_path.exists() and manifest_md_path.stat().st_size > 0
assert {"3.2", "3.3", "3.5", "5.1", "5.2", "5.3", "5.4", "5.5", "5.6", "6.1"}.issubset(set(manifest_df["figure_id"]))
{
    "required_outputs": len(required_outputs),
    "manual_prompt_figures_documented": ["1.1", "1.2", "3.1", "3.4", "4.1", "4.2", "4.3", "4.4"],
    "manifest_rows": len(manifest_df),
    "manifest_csv": str(manifest_csv_path),
    "manifest_md": str(manifest_md_path),
}
